## <center><b> Pre-Processing MST data

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyvital
from dotenv import load_dotenv
import os
from tqdm.notebook import tqdm
import statsmodels.api as sm
from scipy.signal import butter, filtfilt
import time

In [ ]:

# Setup
load_dotenv('vars.env')
DATA_PATH = os.getenv("DATA_PATH")
DEMOGRAPHICS = os.getenv("DEMOGRAPHICS")

SAMPLE_LENGTH = int(os.getenv("SAMPLE_LENGTH"))
SAMPLING_RATE = float(os.getenv("SAMPLING_RATE"))

PLOT_LINKING = os.getenv("PLOT_LINKING") == "True"
PLOT_HR = os.getenv("PLOT_HR") == "True"
PLOT_PP = os.getenv("PLOT_PP") == "True"

FINAL_PLOT = os.getenv("PLOT_FINAL") == "True"

FILTER_BEGIN = os.getenv("FILTER_BEGIN") == "True"
FILTER_HR = os.getenv("FILTER_HR") == "True"
FILTER_PP = os.getenv("FILTER_PP") == "True"

NPY_SAVE = os.getenv("NPY_SAVE") == "True"

In [ ]:
# Collect paths from patient records
patients = []
vital_base = os.path.join(DATA_PATH, 'Vital data')
hs_base    = os.path.join(DATA_PATH, 'HemoSphere data')

for folder in sorted(os.listdir(vital_base)):
    vital_pad = os.path.join(vital_base, folder, 'vital.csv')
    hs_pad    = os.path.join(hs_base,    folder, 'hemosphere.csv')

    if os.path.exists(vital_pad) and os.path.exists(hs_pad):
        patients.append((vital_pad, hs_pad))

print(f"Patients found: {len(patients)}")

In [ ]:
# Loading data
def load_vital_safe(pad, retries=3):
    for i in range(retries):
        try:
            return load_vital(pad)
        except OSError:
            print(f"Time-out, try {i+1}/{retries}...")
            time.sleep(2)
    raise OSError(f"Kan bestand niet lezen: {pad}")

def load_vital(pad):
    df_v = pd.read_csv(pad)
    df_v['Time'] = pd.to_timedelta(df_v['Time'])
    df_v['minutes'] = (df_v['Time'] - df_v['Time'].min()).dt.total_seconds() / 60
    return df_v

def load_hemosphere(pad):
    df_h = pd.read_csv(pad, sep=';')
    df_h = df_h.iloc[1:].reset_index(drop=True)
    df_h['Tijd'] = pd.to_timedelta(df_h['Tijd'].str.strip())
    df_h['SV'] = pd.to_numeric(df_h['SV'].str.strip().replace(r'\\n', np.nan, regex=True), errors='coerce') #verwijderd de \n tekens in de SV kolom.
    df_h = df_h.dropna(subset=['SV']).reset_index(drop=True)
    return df_h

def load_demographics(patient_name, base_pad=DATA_PATH):
    pad = os.path.join(base_pad, DEMOGRAPHICS)
    df  = pd.read_csv(pad, sep=';')
    rij = df[df.iloc[:, 0] == int(patient_name)].copy()
    rij.iloc[:, 2] = (rij.iloc[:, 2] == 'Male').astype(int)
    return rij.iloc[:, 1:5].astype(float)

# ABP resampling
def resample_abp(df):
    ABP = df[['minutes', 'Intellivue/ABP']].dropna().reset_index(drop=True)
    ABP.columns = ['minutes', 'ABP']
    time_diff = ABP['minutes'].diff().dropna() * 60
    sample_freq = 1 / time_diff.mean()
    resample_ABP = np.array(pyvital.resample_hz(ABP['ABP'], sample_freq, 100), dtype=np.float32)
    t_start = ABP['minutes'].min()
    return resample_ABP, t_start

# Linking ABP to SV
def link_abp_sv(df, df_h, resample_ABP, show_plot=PLOT_LINKING):
    time_vital = df['Time'].dt.total_seconds().values
    time_hs = df_h['Tijd'].dt.total_seconds().values
    sv_values = df_h['SV'].values
    
    abp_mask = ~np.isnan(df['Intellivue/ABP'].values)
    t_abp_start = time_vital[abp_mask][0]
    t_abp_end   = time_vital[abp_mask][-1]

    samples_abp, samples_sv, sv_times = [], [], []

    for i in range(len(time_hs)):
        sv_time = time_hs[i]

        if sv_time < t_abp_start or sv_time > t_abp_end:
            continue

        sv_idx = round((sv_time - t_abp_start) * 100)
        start_idx = sv_idx - SAMPLE_LENGTH

        if start_idx < 0 or sv_idx > len(resample_ABP):
            continue

        segment = np.array(resample_ABP[start_idx:sv_idx])
        if len(segment) != SAMPLE_LENGTH:
            continue

        samples_abp.append(segment)
        samples_sv.append(sv_values[i])
        sv_times.append(sv_time)

    samples_abp = np.array(samples_abp, dtype=np.float32)
    samples_sv = np.array(samples_sv)
    sv_times = np.array(sv_times)
    g_indices = ((sv_times - t_abp_start) / 20).astype(int)

    if show_plot:
        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

        for i in range(len(samples_abp)):
            t_seg = np.linspace(sv_times[i] - 20, sv_times[i], SAMPLE_LENGTH) / 3600
            axes[0].plot(t_seg, samples_abp[i], color="#85B7EB", linewidth=0.5)
            axes[0].axvline(sv_times[i] / 3600, color="#E24B4A", linewidth=0.5, alpha=0.3)

        axes[0].set_title("ABP segments with SV timestamps (red)", fontsize=11, fontweight="bold", loc="left")
        axes[0].set_ylabel("ABP (mmHg)")
        axes[0].grid(True, alpha=0.2)

        axes[1].plot(sv_times / 3600, samples_sv, color="#378ADD", linewidth=1.0, marker="o", markersize=3)
        axes[1].set_title("Stroke Volume (SV)", fontsize=11, fontweight="bold", loc="left")
        axes[1].set_ylabel("SV (mL)")
        axes[1].set_xlabel("Clock Time (hours)")
        axes[1].grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

    return samples_abp, samples_sv, sv_times, g_indices, t_abp_start

# Lowess smoothing of SV values
def lowess_smoothing(samples_sv, indices_sv, sampling_rate=1 / 100): #Filter vanuit, van Mierlo
    """
    Performs LOWESS smoothing using the statsmodels.nonparametric.smoothers_lowess.lowess package. If SV values were
    not recorded for > 200s within a single record, the LOWESS algorithm is applied to the slices before and after the
    measurement gap separately.
    :param samples_sv:np.ndarray           SVs at the end of the corresponding ABP segment with shape (nr_of_samples, 1)
    :param indices_sv:np.ndarray           time indices of the SVs with shape (nr_of_samples, 1)
    :param sampling_rate:float             sampling rate of the data in seconds
    :return:
            samples_sv_smoothed:np.ndarray SVs smoothed using LOWESS with shape (nr_of_samples, 1)
    """
    diffs = np.diff(indices_sv.reshape(-1))
    slicing_locs = (np.where(diffs > 200 / sampling_rate)[0] + 1).tolist()  # +1 since we want the slice loc
    slicing_locs = [0] + slicing_locs + [len(indices_sv) + 1]  # add start and end of list
    samples_sv_smoothed = np.array([])
    for slice_index in range(1, len(slicing_locs)):  # loop over the slices
        i_slice = indices_sv[slicing_locs[slice_index - 1]:slicing_locs[slice_index]]
        s_slice = samples_sv[slicing_locs[slice_index - 1]:slicing_locs[slice_index]]
        s_slice_smoothed = sm.nonparametric.lowess(exog=i_slice.reshape([len(i_slice)]),
                                                   endog=s_slice.reshape([len(s_slice)]),
                                                   frac=0.03, return_sorted=False)
        samples_sv_smoothed = np.append(samples_sv_smoothed, s_slice_smoothed)
    return samples_sv_smoothed.reshape(len(samples_sv_smoothed), 1)

def lowess_sv(samples_sv, g_indices): #Application of filter van Mierlo
    indices_sv = (g_indices * 2000).reshape(-1, 1)
    return lowess_smoothing(
        samples_sv=samples_sv.reshape(-1, 1),
        indices_sv=indices_sv,
        sampling_rate=SAMPLING_RATE
    )

# Deletion of first 10 minutes
def filter_begin(segments, indices_to_be_removed, n_segmenten=30, actief=FILTER_BEGIN):
    if not actief:
        return indices_to_be_removed
    for i in range(min(n_segmenten, len(segments))):
        indices_to_be_removed.add(i)
    return indices_to_be_removed

# Blood pressure measurements deletion
def filter_abp_dip(segments, sv_times, resample_ABP, t_abp_start, indices_to_be_removed, venster_s=5, drempel=25, active=True):
    if not active:
        return

    # Rolling mean over het hele continue signaal
    venster = venster_s * 100
    kernel = np.ones(venster) / venster
    rolling_mean = np.convolve(resample_ABP, kernel, mode='same')

    # Globale baseline
    baseline = np.median(rolling_mean)

    # Dip masker op signaalniveau
    dip_mask = rolling_mean < (baseline - drempel)

    # Verwijder elk segment dat ook maar gedeeltelijk overlapt met een dip
    for i, sv_t in enumerate(sv_times):
        seg_einde = round((sv_t - t_abp_start) * 100)
        seg_start = seg_einde - SAMPLE_LENGTH
        seg_start = max(0, seg_start)
        seg_einde = min(len(dip_mask), seg_einde)

        if np.any(dip_mask[seg_start:seg_einde]):
            indices_to_be_removed.add(i)
         
# Noise filtering
def filter_physiological(segments, sv_smoothed, indices_to_be_removed): #Physiologically implausible values, such as ABP < 25 or > 250 mmHg, or SV < 20 or > 200 mL. 
    mask = (
        (segments.min(axis=1) < 25) | (segments.max(axis=1) > 250) |
        (sv_smoothed.flatten() < 20) | (sv_smoothed.flatten() > 200)
    )
    indices_to_be_removed.update(np.where(mask)[0])
   
# Heart rate filtering  
def filter_heartrate(segments, sv_times, indices_to_be_removed, active=FILTER_HR, show_plot=PLOT_HR):
    if not active:
        return

    def bandpass_abp(seg, srate=100, fl=0.5, fh=3.3):
        nyq = srate / 2
        b, a = butter(2, [fl / nyq, fh / nyq], btype='band')
        return filtfilt(b, a, seg)

    hr_list, hr_times = [], []
    t_offset = sv_times[0] / 60 - 20 / 60

    if show_plot:
        fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

    for i, seg in enumerate(segments):
        seg_filtered = bandpass_abp(seg)
        try:
            mins, maxs = pyvital.detect_peaks(seg, srate=100)
        except IndexError:
            indices_to_be_removed.add(i)
            continue
        mins, maxs = np.array(mins), np.array(maxs)

        if show_plot:
            t_start_seg = sv_times[i] / 60 - 20 / 60 - t_offset
            colors = "#E24B4A" if i in indices_to_be_removed else "#85B7EB"
            x_seg = np.linspace(t_start_seg, t_start_seg + 20 / 60, SAMPLE_LENGTH)
            axes[0].plot(x_seg, seg, color=colors, linewidth=0.4, alpha=0.6)
            axes[1].plot(x_seg, seg_filtered, color=colors, linewidth=0.4, alpha=0.6)
            if len(maxs) > 0:
                x_peaks = t_start_seg + maxs / 100 / 60
                axes[1].scatter(x_peaks, seg_filtered[maxs], color="#E8A838", s=8, zorder=5)

        if len(mins) < 2 or len(maxs) < 2:
            indices_to_be_removed.add(i)
            continue

        median_interval = np.median(np.concatenate([np.diff(mins), np.diff(maxs)]))
        hr_bpm = 1 / (median_interval * SAMPLING_RATE) * 60

        if np.isnan(hr_bpm) or not (30 <= hr_bpm <= 180):
            indices_to_be_removed.add(i)

        if not np.isnan(hr_bpm):
            hr_list.append(hr_bpm)
            hr_times.append(sv_times[i] / 60 - t_offset)

    if show_plot:
        axes[0].plot([], [], color="#85B7EB", label="Good segment")
        axes[0].plot([], [], color="#E24B4A", label="Deleted segment")
        axes[0].set_title("Origineel signaal", fontsize=11, fontweight="bold", loc="left")
        axes[0].set_ylabel("ABP (mmHg)")
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.2)

        axes[1].plot([], [], color="#85B7EB", label="Good segment")
        axes[1].plot([], [], color="#E24B4A", label="Deleted segment")
        axes[1].scatter([], [], color="#E8A838", s=8, label="Peak")
        axes[1].set_title("Na bandpass filter (0.5–4 Hz)", fontsize=11, fontweight="bold", loc="left")
        axes[1].set_ylabel("ABP (mmHg)")
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.2)

        axes[2].plot(hr_times, hr_list, color="#378ADD", linewidth=1.0, marker="o", markersize=3)
        axes[2].axhline(30,  color="#E24B4A", linewidth=0.8, linestyle="--", alpha=0.5)
        axes[2].axhline(180, color="#E24B4A", linewidth=0.8, linestyle="--", alpha=0.5, label="Border (30-180 bpm)")
        axes[2].set_title("Heart rate over time", fontsize=11, fontweight="bold", loc="left")
        axes[2].set_ylabel("Heart rate (bpm)")
        axes[2].set_xlabel("Time (min)")
        axes[2].legend(fontsize=9)
        axes[2].grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

# Pulse pressure filtering
def filter_pulse_pressure(segments, sv_times, indices_to_be_removed, active=FILTER_PP, show_plot=PLOT_PP):
    if not active:
        return

    pp_list, pp_times = [], []
    t_offset = sv_times[0] / 60 - 20 / 60

    for i, seg in enumerate(segments):
        try: 
            mins, maxs = pyvital.detect_peaks(seg, srate=100)
        except IndexError:
            indices_to_be_removed.add(i)
            continue
        mins, maxs = np.array(mins), np.array(maxs)

        if len(mins) < 1 or len(maxs) < 2:
            indices_to_be_removed.add(i)
            pp_list.append(np.nan)
        else:
            min_vals = seg[mins]
            max_vals = seg[maxs]
            pps = [max_vals[m + 1] - min_vals[m] for m in range(len(min_vals)) if m + 1 < len(max_vals)]

            if not pps:
                indices_to_be_removed.add(i)
                pp_list.append(np.nan)
            else:
                pp_mean = np.mean(pps)
                pp_list.append(pp_mean)
                if pp_mean < 20:
                    indices_to_be_removed.add(i)

        pp_times.append(sv_times[i] / 60 - t_offset)

    if show_plot:
        pp_array = np.array(pp_list)
        fig, ax = plt.subplots(figsize=(16, 4))

        colors = ["#E24B4A" if i in indices_to_be_removed else "#378ADD" for i in range(len(segments))]
        ax.scatter(pp_times, pp_array, c=colors, s=15, zorder=5)
        ax.plot(pp_times, pp_array, color="#85B7EB", linewidth=0.6, alpha=0.5)
        ax.axhline(20, color="#E24B4A", linewidth=1.0, linestyle="--", label="Border (20 mmHg)")
        ax.set_title("Pulse pressure over time", fontsize=11, fontweight="bold", loc="left")
        ax.set_ylabel("Gemiddelde PP per segment (mmHg)")
        ax.set_xlabel("Time (min)")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()
        
# Segments deletion
def delete_segments(segments, sv, g_indices, sv_times, indices_to_be_removed, patient_name="", show_plot=FINAL_PLOT):
    mask = np.ones(len(segments), dtype=bool)
    mask[list(indices_to_be_removed)] = False

    if show_plot:
        fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

        # Voor filtering — ABP
        for i in range(len(segments)):
            x = np.arange(i * SAMPLE_LENGTH, (i + 1) * SAMPLE_LENGTH) / 100 / 60
            color = "#E24B4A" if not mask[i] else "#85B7EB"
            axes[0].plot(x, segments[i], color=color, linewidth=0.4, alpha=0.8)

        axes[0].plot([], [], color="#85B7EB", label="Good segment")
        axes[0].plot([], [], color="#E24B4A", label="Removed segment")
        axes[0].set_title(f"{patient_name} — Voor filtering", fontsize=11, fontweight="bold", loc="left")
        axes[0].set_ylabel("ABP (mmHg)")
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.2)

        # Na filtering — ABP
        signal_na = np.full(len(segments) * SAMPLE_LENGTH, np.nan)
        for i in np.where(mask)[0]:
            signal_na[i * SAMPLE_LENGTH:(i + 1) * SAMPLE_LENGTH] = segments[i]

        x_na = np.arange(len(signal_na)) / 100 / 60
        axes[1].plot(x_na, signal_na, color="#85B7EB", linewidth=0.4)
        axes[1].set_title(f"{patient_name} — After filtering", fontsize=11, fontweight="bold", loc="left")
        axes[1].set_ylabel("ABP (mmHg)")
        axes[1].grid(True, alpha=0.2)

        # SV — voor en na filtering
        x_sv = np.arange(len(sv)) * SAMPLE_LENGTH / 100 / 60
        axes[2].scatter(x_sv, sv, color=["#E24B4A" if not mask[i] else "#85B7EB" for i in range(len(sv))], s=10, zorder=5)
        axes[2].plot(x_sv[mask], sv[mask], color="#378ADD", linewidth=0.8)
        axes[2].set_title(f"{patient_name} — SV (blue = good, red = removed)", fontsize=11, fontweight="bold", loc="left")
        axes[2].set_ylabel("SV (mL)")
        axes[2].set_xlabel("Time (min)")
        axes[2].grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

    return segments[mask], sv[mask], g_indices[mask]

# Data saving in npy files
def save_data(segments, sv, demographics, ids, base_pad=DATA_PATH, active=NPY_SAVE):
    if not active:
        return
    
    os.makedirs(os.path.join(base_pad, 'npy_files'), exist_ok=True)
    np.save(os.path.join(base_pad, 'npy_files', 'np_w_finetune.npy'),  segments)
    np.save(os.path.join(base_pad, 'npy_files', 'np_sv_finetune.npy'), sv)
    np.save(os.path.join(base_pad, 'npy_files', 'np_a_finetune.npy'),  demographics)
    np.save(os.path.join(base_pad, 'npy_files', 'np_c_finetune.npy'), ids)
    print(f'Saved in: {os.path.join(base_pad, "npy_files")}')
    print(f'np_w: {segments.shape}')
    print(f'np_sv: {sv.shape}')
    print(f'np_a: {demographics.shape}')
    print(f'np_c: {ids.shape}')
    
# Plays all definitions
def process_patient(vital_pad, hs_pad):
    patient_name = os.path.basename(os.path.dirname(vital_pad)).replace('_vital', '').replace('MARTINI_', '')
    print(f'Processing: {vital_pad}')

    df_v = load_vital_safe(vital_pad)
    df_h = load_hemosphere(hs_pad)
    demographics = load_demographics(patient_name)

    resample_ABP, t_start = resample_abp(df_v)

    segments, sv, sv_times, g_indices, t_abp_start = link_abp_sv(df_v, df_h, resample_ABP)
    print(f'Paired: {len(segments)}')

    sv_smoothed = lowess_sv(sv, g_indices)

    indices_to_be_removed = set()

    steps = [
        ('Begin', lambda: filter_begin(segments, indices_to_be_removed)),
        ('ABP dip',      lambda: filter_abp_dip(segments, sv_times, resample_ABP, t_abp_start, indices_to_be_removed)),
        ('Physiological', lambda: filter_physiological(segments, sv_smoothed, indices_to_be_removed)),
        ('Heart rate', lambda: filter_heartrate(segments, sv_times, indices_to_be_removed)),
        ('Pulse pressure', lambda: filter_pulse_pressure(segments, sv_times, indices_to_be_removed)),
    ]

    with tqdm(total=len(steps), desc='Filtering') as pbar: #pbar en vooruitgangs weergave.
        for naam, stap in steps:
            voor = len(indices_to_be_removed)
            stap()
            na = len(indices_to_be_removed)
            pbar.set_postfix({naam: f'+{na - voor}'})
            pbar.update(1)

    filtered_segments, filtered_sv, filtered_indices = delete_segments(segments, sv_smoothed.flatten(), sv_times, g_indices, indices_to_be_removed, patient_name=patient_name)

    print(f'After filtering:{len(filtered_segments)}')
    print(f'Total removed: {len(indices_to_be_removed)}')
    
    return filtered_segments, filtered_sv, filtered_indices


In [ ]:
%matplotlib inline
all_segments = []
all_sv = []
all_indices = []
all_demographics = []
all_ids = []

with tqdm(total=len(patients), desc='Patiënten verwerken') as pbar:
    for vital_pad, hs_pad in patients:
        seg, sv, idx = process_patient(vital_pad, hs_pad)
        patient_name = os.path.basename(os.path.dirname(vital_pad)).replace('_vital', '').replace('MARTINI_', '')
        demographics = load_demographics(patient_name)

        all_segments.append(seg)
        all_sv.append(sv)
        all_indices.append(idx)
        all_demographics.append(np.tile(demographics.values, (len(seg), 1)))
        all_ids.append(np.full(len(seg), int(patient_name)))
        pbar.update(1)

all_segments = np.concatenate(all_segments, axis=0)
all_sv = np.concatenate(all_sv, axis=0)
all_indices = np.concatenate(all_indices, axis=0)
all_demographics = np.concatenate(all_demographics, axis=0)
all_ids = np.concatenate(all_ids, axis=0)

print(f'Totaal segmenten: {len(all_segments)}')
print(f'Totaal SV:        {len(all_sv)}')
print(f'Demographics:     {all_demographics.shape}')
print(f'IDs:              {all_ids.shape}')

save_data(all_segments, all_sv, all_demographics, all_ids)